# <font color="brown">✨&nbsp;&nbsp;Introduction</font>

Maybe you know the Mastermind game. You must guess a color combination and you have as feedback the number of pegs that have the right color and the right position and the number of pegs with the right color but a wrong position.

Here we will try to guess the correct combination with a Genetic Algorithm.

We will allow duplicated colors.

Contrary to the board game, we will not enforce a maximum amount of trial as genetic algorithms rely on evolving solutions over multiple generations and as such it is not really relevant to consider trials (what would be generations and populations then? Populations of one individual doesn't make much sense).

<img src="https://upload.wikimedia.org/wikipedia/commons/2/2d/Mastermind.jpg" width="300px">

## <font color="brown">🎳&nbsp;&nbsp;Define Game</font>

In [ ]:
N_PEGS = 10 # Number of pegs to guess
N_COLORS = 10 # Number of different colors in the game

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Generate a color palette
palette = sns.color_palette(palette="cubehelix", n_colors=N_COLORS)

# Function to display the board corresponding to a given chromosome
def display_board(chromosome):
  board = np.hstack([
      np.hstack([
          np.full((20, 2, 3), (1, 1, 1)),
          np.full((20, 20, 3), palette[peg]),
      ]) for i, peg in enumerate(chromosome)
  ])

  plt.figure(dpi=90)
  plt.axis('off')
  plt.imshow(board)
  plt.show()

In [ ]:
# See all colors
display_board([i for i in np.arange(N_COLORS)])

## <font color="brown">🎲&nbsp;&nbsp;Generate randomly a combination to guess</font>

In [ ]:
import numpy as np

# Generate the combination to guess
correct_combination = np.random.randint(0, N_COLORS, N_PEGS)

# Display it
display_board(correct_combination)

# Create a dictionnary counting the number of each color in the solution
correct_color_count = {
    color: np.count_nonzero(correct_combination == color)
    for color in np.unique(correct_combination)
}

# Print the count of each color
print(correct_color_count)

## <font color="brown">🏋️&nbsp;&nbsp;Define fitness function</font>

In [ ]:
# Score for guessing exactly a peg
SCORE_CORRECT_PEGS = 5

# Score for guessing correctly a color
SCORE_CORRECT_COLOR = 1

# The maximum score (score of the correct combination).
# Note that for a peg guessed exactly, we score points both for
# guessing exactly and for having the right color.
MAX_SCORE = SCORE_CORRECT_PEGS * N_PEGS + SCORE_CORRECT_COLOR * N_PEGS

def score_chromosome(chromosome):
    # Colors score
    colors_score = 0

    for color, color_count in correct_color_count.items():
        # We win points for each colors guessed correctly
        # but not more than the number of pegs of this colors
        # in the correct combination
        colors_score += min(np.count_nonzero(chromosome == color), color_count)

    colors_score *= SCORE_CORRECT_COLOR

    # Correct pegs score
    correct_peg_score = np.count_nonzero(np.array(chromosome) == correct_combination)
    correct_peg_score *= SCORE_CORRECT_PEGS

    return colors_score + correct_peg_score

def fitness_function(ga_instance, solution, solution_idx):
    return score_chromosome(solution)

## <font color="brown">☎️&nbsp;&nbsp;Define Callbacks </font>

In [ ]:
def on_generation(ga_instance):
    solution, solution_fitness, solution_idx = ga_instance.best_solution()
    print("Generation: ", ga_instance.generations_completed, ". Fitness: ", solution_fitness)
    solution_image = display_board(solution)

## <font color="brown">🚀&nbsp;&nbsp;Run genetic algorithm</font>

In [ ]:
#%pip install pygad

In [ ]:
import pygad

ga_instance = pygad.GA(
    sol_per_pop=50,
    num_genes=N_PEGS,
    num_generations=1000,
    num_parents_mating=4,
    fitness_func=fitness_function,
    gene_type=int,
    init_range_low=0,
    init_range_high=(N_COLORS - 1),
    on_generation=on_generation,
    mutation_type="random",
    mutation_probability=0.10,
    mutation_by_replacement=True,
    random_mutation_min_val=0.0,
    random_mutation_max_val=N_COLORS,
    crossover_type="uniform",
    crossover_probability=0.8,
    parent_selection_type="rws", # roulette wheel selection
    stop_criteria=f"reach_{MAX_SCORE}"
)

ga_instance.run()

## <font color="brown">🤪&nbsp;&nbsp;Comparison with brute-force search</font>

Here you can try to find the correct combination with a brute-force approach. See how much more time it would take to find it that way (excepted if you're lucky and the correct combination is in the first solutions tried...).

Don't run it until the solution is found.

In [ ]:
import itertools

for combination in itertools.product(np.arange(N_COLORS), repeat=N_PEGS):
    display_board(combination)

    if score_chromosome(combination) == MAX_SCORE:
        break

## <font color="brown">🎯&nbsp;&nbsp; Your task</font>

- Adapt the code to be able to find a sentence of 28 characters instead (only all caps A-Z characters and blank space).

## <font color="brown">📖&nbsp;&nbsp; Report</font>

1. With your code, what would be the chromosome for the sentence "METHINKS IT IS LIKE A WEASEL"?

**Submit your notebook as well**

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pygad
import string

TARGET_SENTENCE = "METHINKS IT IS LIKE A WEASEL"
N_CHARS = len(TARGET_SENTENCE)
CHAR_SET_SIZE = 27

# Mapping between chars and integers
char_mapping = list(string.ascii_uppercase) + [' ']
int_mapping = {char: idx for idx, char in enumerate(char_mapping)}
print(f"Char mapping: {char_mapping}")
print(f"Int mapping: {int_mapping}")

# Convert the target sentence to a numeric representation
target_numeric = np.array([int_mapping[char] for char in TARGET_SENTENCE])

# Create a dict counting the number of each char in the target
target_char_count = {
    char_idx: np.count_nonzero(target_numeric == char_idx)
    for char_idx in np.unique(target_numeric)
}

# Create a list to store fitness history
fitness_history = []

def display_chromosome(chromosome):
    sentence = ''.join([char_mapping[gene] for gene in chromosome])
    print(f"Raw chromosome: {chromosome}")
    print(f"Formatted chromosome: {sentence}")

SCORE_CORRECT_CHAR_POS = 5
SCORE_CORRECT_CHAR = 1
MAX_SCORE_CHAR = SCORE_CORRECT_CHAR_POS * N_CHARS + SCORE_CORRECT_CHAR * N_CHARS

# Fitness function
def score_chromosome(chromosome):
    char_score = 0
    # Char score
    for char_idx, char_count in target_char_count.items():
        char_score += min(np.count_nonzero(chromosome == char_idx), char_count)
    char_score *= SCORE_CORRECT_CHAR
    
    # Position score
    position_score = np.count_nonzero(chromosome == target_numeric)
    position_score *= SCORE_CORRECT_CHAR_POS
    
    return char_score + position_score

def fitness_function(ga_instance, solution, solution_idx):
    return score_chromosome(solution)

def on_generation(ga_instance):
    solution, solution_fitness, solution_idx = ga_instance.best_solution()
    
    # Store the best fitness for this generation
    fitness_history.append(solution_fitness)
    
    if ga_instance.generations_completed % 50 == 0 or solution_fitness == MAX_SCORE_CHAR:
        print("Generation: ", ga_instance.generations_completed, " Fitness: ", solution_fitness)
        solution_image = display_chromosome(solution)

NUM_GENERATIONS = 1000
POPULATION = 300
PARENTS_MATING = 20
MUTATION_PROBA = 0.05
CROSSOVER_PROBA = 0.9

gene_space = [list(range(CHAR_SET_SIZE))] * N_CHARS

# Create and run the genetic algorithm
ga_instance = pygad.GA(
    sol_per_pop=POPULATION,
    num_genes=N_CHARS,
    num_generations=NUM_GENERATIONS,
    num_parents_mating=PARENTS_MATING,
    fitness_func=fitness_function,
    gene_space=gene_space,
    gene_type=int,
    #init_range_low=0,
    #init_range_high=(CHAR_SET_SIZE - 1),
    on_generation=on_generation,
    mutation_type="random",
    mutation_probability=MUTATION_PROBA,
    mutation_by_replacement=True,
    random_mutation_min_val=0,
    random_mutation_max_val=CHAR_SET_SIZE,
    crossover_type="uniform",
    crossover_probability=CROSSOVER_PROBA,
    parent_selection_type="sss",
    #keep_parents=1,
    stop_criteria=f"reach_{MAX_SCORE_CHAR}"
)

# Run the genetic algorithm
print(f"Attempting to find: '{TARGET_SENTENCE}'")
print("-" * 50)
ga_instance.run()

# Display the final solution
solution, solution_fitness, solution_idx = ga_instance.best_solution()
print("\nFinal Solution:")
display_chromosome(solution)

# Calculate how many generations it took
generations_completed = ga_instance.generations_completed
print(f"\nSolution found in {generations_completed} generations")
print(f"Fitness: {solution_fitness} out of {MAX_SCORE_CHAR}")

# Create an improved fitness plot
plt.figure(figsize=(10, 6))

# Plot the fitness history with appropriate x-values (generations)
generations = range(1, len(fitness_history) + 1)
plt.plot(generations, fitness_history, 'b-', linewidth=2, label='Best Fitness')

# Add a horizontal line for the maximum possible fitness
plt.axhline(y=MAX_SCORE_CHAR, color='r', linestyle='--', 
            alpha=0.7, label=f'Maximum Fitness ({MAX_SCORE_CHAR})')

# Customize the plot
plt.title('Fitness Evolution During Genetic Algorithm Execution', fontsize=14)
plt.xlabel('Generation', fontsize=12)
plt.ylabel('Fitness Score', fontsize=12)
plt.grid(True, alpha=0.3, linestyle='--')

plt.xlim(0, len(fitness_history) + 5)
plt.ylim(min(fitness_history) * 0.9, MAX_SCORE_CHAR * 1.05)

plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

Error creating the initial population:

When the parameter 'initial_population' is None, then the 2 parameters 'sol_per_pop' and 'num_genes' cannot be None too.
There are 2 options to prepare the initial population:
1) Assinging the initial population to the 'initial_population' parameter. In this case, the values of the 2 parameters sol_per_pop and num_genes will be deduced.
2) Assign integer values to the 'sol_per_pop' and 'num_genes' parameters so that PyGAD can create the initial population automatically.
Traceback (most recent call last):
  File "/usr/local/Caskroom/miniforge/base/envs/data_science_2025/lib/python3.9/site-packages/pygad/pygad.py", line 409, in __init__
    raise TypeError("Error creating the initial population:\n\nWhen the parameter 'initial_population' is None, then the 2 parameters 'sol_per_pop' and 'num_genes' cannot be None too.\nThere are 2 options to prepare the initial population:\n1) Assinging the initial population to the 'initial_population' parameter. I

Char mapping: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', ' ']
Int mapping: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'J': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 13, 'O': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'U': 20, 'V': 21, 'W': 22, 'X': 23, 'Y': 24, 'Z': 25, ' ': 26}


TypeError: Error creating the initial population:

When the parameter 'initial_population' is None, then the 2 parameters 'sol_per_pop' and 'num_genes' cannot be None too.
There are 2 options to prepare the initial population:
1) Assinging the initial population to the 'initial_population' parameter. In this case, the values of the 2 parameters sol_per_pop and num_genes will be deduced.
2) Assign integer values to the 'sol_per_pop' and 'num_genes' parameters so that PyGAD can create the initial population automatically.